**TIP:** You can save your work offline by using `File -> Download`.
Later you can upload the notebook back into the workspace.

## Setup

In [1]:
# SETUP CELL
%pip -q install numpy sympy matplotlib ipywidgets pandas plotly anywidget scipy

Note: you may need to restart the kernel to use updated packages.


In [2]:
# SETUP CELL
import src_path
from gu_toolkit import *
from helpers.Fourier_02_helper import *

# Fourier coefficients of the square wave via `NIntegrate`

In `Fourier_02` you adjusted sine coefficients by hand.
In this notebook we will **compute** them numerically.

We work with the 1-periodic square wave
$$
\\mathrm{Sq}(x)=
\begin{cases}
-1 &\text{if } x<0,\\
1 &\text{if } x>0.
\end{cases}
$$

Because this function is odd, its Fourier series uses only sine terms:
$$
S_N(x)=\sum_{n=1}^{N} b_n\sin(2\pi n x).
$$

## The square wave

In [3]:
@NamedFunction
def Sq(x):
    return sign(sin(2 * pi * x))

fig_square = Figure(x_range=(-1 / 2, 1 / 2), y_range=(-2, 2))
fig_square.title = r"The square wave $\mathrm{Sq}(x)$"
display(fig_square)

with fig_square:
    plot(Sq(x), x, id="Sq", label=r"$\mathrm{Sq}(x)$")

C:\Users\guraltsev\AppData\Local\Temp\ipykernel_27464\4046956186.py:5: DeprecationWarning: Figure(x_range=...) is deprecated; use Figure(default_x_range=...) instead.
  fig_square = Figure(x_range=(-1 / 2, 1 / 2), y_range=(-2, 2))
C:\Users\guraltsev\AppData\Local\Temp\ipykernel_27464\4046956186.py:5: DeprecationWarning: Figure(y_range=...) is deprecated; use Figure(default_y_range=...) instead.
  fig_square = Figure(x_range=(-1 / 2, 1 / 2), y_range=(-2, 2))


OneShotOutput()

## Step 1. Compute the sine coefficients

On the interval $[-\tfrac12,\tfrac12]$, the sine coefficients are
$$
b_n = 2\int_{-1/2}^{1/2} \mathrm{Sq}(x)\sin(2\pi n x)\,dx.
$$

The next cell uses `NIntegrate` to compute these numbers.
Start with a moderate value such as `Nmax = 15`.

In [4]:
Nmax = 15

b_coeff = np.zeros(Nmax + 1)
for n in range(1, Nmax + 1):
    b_coeff[n] = NIntegrate(
        2 * Sq(x) * sin(2 * pi * n * x),
        (x, -1 / 2, 1 / 2),
    )

coeff_table = pd.DataFrame(
    {
        "n": np.arange(1, Nmax + 1),
        "b_n": b_coeff[1:],
        "|b_n|": np.abs(b_coeff[1:]),
    }
)
display(coeff_table.round(6))

,n,b_n,|b_n|
0,1,1.273240,1.273240
1,2,-0.000000,0.000000
2,3,0.424413,0.424413
3,4,-0.000000,0.000000
4,5,0.254648,0.254648
5,6,0.000000,0.000000
6,7,0.181891,0.181891
7,8,-0.000000,0.000000
8,9,0.141471,0.141471
9,10,-0.000000,0.000000


What do you notice?

- The odd coefficients are large.
- The even coefficients are essentially `0`.
- This matches the symmetry you observed in `Fourier_02`.

## Step 2. Build the partial sums

In [5]:
coeff_for_plot = b_coeff.copy()
coeff_for_plot[np.abs(coeff_for_plot) < 1e-10] = 0.0

S = [0]
for n in range(1, Nmax + 1):
    S.append(S[-1] + coeff_for_plot[n] * sin(2 * pi * n * x))

display(Latex(r"$S_N(x)=\sum_{n=1}^{N} b_n\sin(2\pi n x)$"))

<IPython.core.display.Latex object>

## Step 3. Plot the approximations

The legend lets you turn individual curves on and off.
Compare the square wave with a few partial sums.

In [6]:
Nshow = [1, 3, 5, 9, Nmax]

fig = Figure(x_range=(-1 / 2, 1 / 2), y_range=(-2.5, 2.5))
fig.title = r"Square wave and Fourier approximations"
display(fig)

with fig:
    plot(Sq(x), x, id="Sq(x)", label=r"$\mathrm{Sq}(x)$")
    for n in Nshow:
        plot(S[n], x, id=f"S{n}", label=rf"$S_{{{n}}}(x)$")

fig.info(MaxDistanceCard(x, Sq(x), S[Nmax]), id="Sq_minus_SN_max")
fig.info(AvgDistanceCard(x, Sq(x), S[Nmax]), id="Sq_minus_SN_avg")

C:\Users\guraltsev\AppData\Local\Temp\ipykernel_27464\3839949686.py:3: DeprecationWarning: Figure(x_range=...) is deprecated; use Figure(default_x_range=...) instead.
  fig = Figure(x_range=(-1 / 2, 1 / 2), y_range=(-2.5, 2.5))
C:\Users\guraltsev\AppData\Local\Temp\ipykernel_27464\3839949686.py:3: DeprecationWarning: Figure(y_range=...) is deprecated; use Figure(default_y_range=...) instead.
  fig = Figure(x_range=(-1 / 2, 1 / 2), y_range=(-2.5, 2.5))


OneShotOutput()

## Step 4. Plot the error

The next plot shows the difference $\mathrm{Sq}(x)-S_N(x)$.
Look especially near the jump at $x=0$.

In [7]:
fig_error = Figure(x_range=(-1 / 2, 1 / 2), y_range=(-1.5, 1.5))
fig_error.title = rf"Difference $\mathrm{{Sq}}(x)-S_{{{Nmax}}}(x)$"
display(fig_error)

with fig_error:
    plot(
        Sq(x) - S[Nmax],
        x,
        id="error",
        label=rf"$\mathrm{{Sq}}(x)-S_{{{Nmax}}}(x)$",
    )

C:\Users\guraltsev\AppData\Local\Temp\ipykernel_27464\3234623905.py:1: DeprecationWarning: Figure(x_range=...) is deprecated; use Figure(default_x_range=...) instead.
  fig_error = Figure(x_range=(-1 / 2, 1 / 2), y_range=(-1.5, 1.5))
C:\Users\guraltsev\AppData\Local\Temp\ipykernel_27464\3234623905.py:1: DeprecationWarning: Figure(y_range=...) is deprecated; use Figure(default_y_range=...) instead.
  fig_error = Figure(x_range=(-1 / 2, 1 / 2), y_range=(-1.5, 1.5))


OneShotOutput()

## Explore

- Change `Nmax` and rerun the last cells.
- Try plotting only the odd partial sums.
- Compare the numerical coefficients with the exact formula
  $$
  b_n = \frac{4}{\pi n}\quad\text{for odd }n,
  \qquad b_n = 0\quad\text{for even }n.
  $$
- Why does the **average** error get small while the **maximum** error stays large near the jump?